# Notebook 02 — Beam Tuning with Bayesian Optimisation

**CAS 2026 · AI for Medical Accelerators**

In this notebook you will:
1. Build a simple proton therapy transport line in **Cheetah** (PyTorch-based differentiable simulator)
2. Define a beam quality objective (minimise RMS spot size at the monitor)
3. Tune quadrupole strengths with **Bayesian optimisation** (scikit-optimize)
4. Compare BO against random search and gradient descent (using Cheetah's differentiability)
5. Visualise the convergence and the optimised beam spot

**References**
- Cheetah simulator: [github.com/desy-ml/cheetah](https://github.com/desy-ml/cheetah) · Kaiser et al., PRAB 27, 054601 (2024)
- For complex lattices: [xsuite](https://github.com/xsuite/xsuite) (CERN tracking framework)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aghribi/cas2026-ai-medical-accelerators/blob/main/notebooks/02_beam_tuning/notebook.ipynb)

In [1]:
# Run this cell only on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q cheetah-accelerator scikit-optimize plotly

In [2]:
import plotly.io as pio; pio.renderers.default = "notebook_connected"
import time
import numpy as np
import torch
import cheetah
from skopt import gp_minimize
from skopt.space import Real
from skopt.plots import plot_convergence
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print(f'torch {torch.__version__}, cheetah OK')

torch 2.6.0, cheetah OK


## 1 · Build a proton therapy transport line in Cheetah

We model a simplified section of a proton therapy beam transport line:
a **FODO-like doublet** (focusing-defocusing quadrupole pair) followed by a diagnostic screen.

The goal: find k₁ values for the two quadrupoles that minimise the RMS beam size at the screen.

In [3]:
segment = cheetah.Segment(elements=[
    cheetah.Drift(length=torch.tensor(0.5)),
    cheetah.Quadrupole(length=torch.tensor(0.1), name="QF"),   # focusing
    cheetah.Drift(length=torch.tensor(1.0)),
    cheetah.Quadrupole(length=torch.tensor(0.1), name="QD"),   # defocusing
    cheetah.Drift(length=torch.tensor(0.5)),
    cheetah.Screen(name="monitor"),
])

# 150 MeV proton beam: total energy = 150 MeV + 938.272 MeV (rest mass)
PROTON_REST_MASS_MEV = 938.272
KINETIC_ENERGY_MEV   = 150.0
TOTAL_ENERGY_EV      = (KINETIC_ENERGY_MEV + PROTON_REST_MASS_MEV) * 1e6

beam_in = cheetah.ParameterBeam.from_twiss(
    energy=torch.tensor(TOTAL_ENERGY_EV),
    beta_x=torch.tensor(1.0),         # Twiss beta in m
    beta_y=torch.tensor(1.0),
    emittance_x=torch.tensor(1e-6),   # geometric emittance m·rad
    emittance_y=torch.tensor(1e-6),
    species=cheetah.Species("proton"),
)

print(f'Beam energy    : {KINETIC_ENERGY_MEV} MeV kinetic')
print(f'Beam size (in) : σ_x={beam_in.sigma_x.item()*1e3:.2f} mm,  σ_y={beam_in.sigma_y.item()*1e3:.2f} mm')

Beam energy    : 150.0 MeV kinetic
Beam size (in) : σ_x=1.00 mm,  σ_y=1.00 mm


## 2 · Define the objective function

The objective is the total RMS beam size at the screen: σ_x + σ_y.
We want to **minimise** this — a smaller, well-focused spot is better.

In [4]:
call_count = 0  # track number of simulator calls

def objective(params: list[float]) -> float:
    """Evaluate beam size at monitor for given (k1_QF, k1_QD)."""
    global call_count
    call_count += 1

    k1_qf, k1_qd = params
    # dtype=float32 required: Cheetah element lengths are float32 and
    # base_rmatrix requires all tensors to share the same dtype.
    segment.QF.k1 = torch.tensor(k1_qf, dtype=torch.float32)
    segment.QD.k1 = torch.tensor(k1_qd, dtype=torch.float32)

    beam_out = segment.track(beam_in)
    result = (beam_out.sigma_x + beam_out.sigma_y).item()
    # Return large fallback for diverged beams (NaN sigma = unphysical configuration)
    return result if np.isfinite(result) else 1.0   # m

# Quick sanity check
print(f'Zero quads → σ = {objective([0.0, 0.0])*1e3:.2f} mm (no focusing)')
print(f'Calls so far: {call_count}')

Zero quads → σ = 4.83 mm (no focusing)
Calls so far: 1


## 3 · Compare three strategies: random search, grid scan, Bayesian optimisation

All three use the same evaluation budget. The key question: who finds the best focus with fewest calls?

### 3a · Random search (baseline)

In [5]:
N_CALLS = 40       # evaluation budget
K1_MIN  = -10.0    # T/m² — physically reasonable for 150 MeV protons
K1_MAX  =  10.0

rng = np.random.default_rng(42)
random_params = rng.uniform(K1_MIN, K1_MAX, size=(N_CALLS, 2))

random_results = []
call_count = 0
for params in random_params:
    random_results.append(objective(list(params)))

best_random = min(random_results)
best_random_idx = np.argmin(random_results)
print(f'Random search: best σ = {best_random*1e3:.2f} mm')
print(f'  at k1_QF = {random_params[best_random_idx,0]:.2f},  k1_QD = {random_params[best_random_idx,1]:.2f} T/m²')

Random search: best σ = 4.41 mm
  at k1_QF = -8.12,  k1_QD = 9.51 T/m²


### 3b · Bayesian optimisation (Gaussian Process surrogate)

In [6]:
call_count = 0
space = [
    Real(K1_MIN, K1_MAX, name="k1_QF"),
    Real(K1_MIN, K1_MAX, name="k1_QD"),
]

t0 = time.perf_counter()
bo_result = gp_minimize(
    func=objective,
    dimensions=space,
    n_calls=N_CALLS,
    n_initial_points=8,   # random exploration before GP takes over
    random_state=42,
    verbose=False,
)
t_bo = time.perf_counter() - t0

print(f'Bayesian opt: best σ = {bo_result.fun*1e3:.2f} mm  (wall time: {t_bo:.1f} s)')
print(f'  at k1_QF = {bo_result.x[0]:.2f},  k1_QD = {bo_result.x[1]:.2f} T/m²')
print(f'  total calls: {call_count}')

Bayesian opt: best σ = 4.36 mm  (wall time: 4.8 s)
  at k1_QF = 7.41,  k1_QD = -10.00 T/m²
  total calls: 40


### 3c · Gradient descent through Cheetah

Cheetah is **differentiable** — we can compute ∂σ/∂k₁ using PyTorch autograd and use gradient descent directly. This only works because Cheetah is a PyTorch module.

In [7]:
# Reset quads to zero and optimise with Adam
segment.QF.k1 = torch.nn.Parameter(torch.tensor(0.0))
segment.QD.k1 = torch.nn.Parameter(torch.tensor(0.0))

optimizer_gd = torch.optim.Adam([segment.QF.k1, segment.QD.k1], lr=1.0)

gd_losses = []
for step in range(N_CALLS):           # same budget as BO
    optimizer_gd.zero_grad()
    beam_out = segment.track(beam_in)
    loss = beam_out.sigma_x + beam_out.sigma_y
    loss.backward()
    optimizer_gd.step()
    gd_losses.append(loss.item())

best_gd = min(gd_losses)
print(f'Gradient descent: best σ = {best_gd*1e3:.2f} mm')
print(f'  k1_QF = {segment.QF.k1.item():.2f},  k1_QD = {segment.QD.k1.item():.2f} T/m²')
print()
print('Note: gradient descent requires differentiability — not possible with a real machine.')
print('      Bayesian optimisation is model-free and works on real hardware.')

Gradient descent: best σ = 4.83 mm
  k1_QF = 0.00,  k1_QD = 0.00 T/m²

Note: gradient descent requires differentiability — not possible with a real machine.
      Bayesian optimisation is model-free and works on real hardware.


## 4 · Visualise convergence

In [8]:
# Running minimum for each method
bo_curve     = np.minimum.accumulate(bo_result.func_vals) * 1e3     # mm
random_curve = np.minimum.accumulate(random_results) * 1e3          # mm
gd_curve     = np.minimum.accumulate(gd_losses) * 1e3               # mm

iters = np.arange(1, N_CALLS + 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=iters, y=bo_curve, name='Bayesian optimisation',
    line=dict(color='#22d3ee', width=2.5)))
fig.add_trace(go.Scatter(x=iters, y=gd_curve, name='Gradient descent (Cheetah autograd)',
    line=dict(color='#a78bfa', width=2.0, dash='dash')))
fig.add_trace(go.Scatter(x=iters, y=random_curve, name='Random search',
    line=dict(color='#f87171', width=1.5, dash='dot')))

fig.update_layout(
    title='Convergence: beam size minimisation (40 evaluations each)',
    xaxis_title='Number of evaluations',
    yaxis_title='Best σ_x + σ_y  (mm)',
    paper_bgcolor='#0d1b2a', plot_bgcolor='#111f30',
    font=dict(color='#e2e8f0'),
    legend=dict(bgcolor='rgba(0,0,0,0)'),
)
fig.update_xaxes(showgrid=True, gridcolor='#1e3048')
fig.update_yaxes(showgrid=True, gridcolor='#1e3048')
fig.show()

## 5 · Visualise the optimised beam

In [9]:
# After gradient-descent, k1 is an nn.Parameter — use .data to update without re-registering
segment.QF.k1.data = torch.tensor(0.0, dtype=torch.float32)
segment.QD.k1.data = torch.tensor(0.0, dtype=torch.float32)
beam_zero = segment.track(beam_in)

segment.QF.k1.data = torch.tensor(bo_result.x[0], dtype=torch.float32)
segment.QD.k1.data = torch.tensor(bo_result.x[1], dtype=torch.float32)
beam_opt  = segment.track(beam_in)

# Sample a ParticleBeam to visualise the spot
pbeam_in = cheetah.ParticleBeam.from_twiss(
    energy=torch.tensor(TOTAL_ENERGY_EV),
    beta_x=torch.tensor(1.0), beta_y=torch.tensor(1.0),
    emittance_x=torch.tensor(1e-6), emittance_y=torch.tensor(1e-6),
    num_particles=5_000,
    species=cheetah.Species("proton"),
)

segment.QF.k1.data = torch.tensor(0.0, dtype=torch.float32)
segment.QD.k1.data = torch.tensor(0.0, dtype=torch.float32)
pbeam_zero = segment.track(pbeam_in)

segment.QF.k1.data = torch.tensor(bo_result.x[0], dtype=torch.float32)
segment.QD.k1.data = torch.tensor(bo_result.x[1], dtype=torch.float32)
pbeam_opt = segment.track(pbeam_in)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['No tuning (k₁ = 0)', f'BO-optimised (σ = {bo_result.fun*1e3:.1f} mm)'])

for col, pbeam in enumerate([pbeam_zero, pbeam_opt], 1):
    x = pbeam.x.detach().numpy() * 1e3   # mm
    y = pbeam.y.detach().numpy() * 1e3
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='markers',
        marker=dict(color='#22d3ee', size=2, opacity=0.3),
        name='Particle' if col == 1 else 'Optimised', showlegend=False,
    ), row=1, col=col)

fig.update_xaxes(title_text='x (mm)', showgrid=True, gridcolor='#1e3048')
fig.update_yaxes(title_text='y (mm)', showgrid=True, gridcolor='#1e3048')
fig.update_layout(
    height=420,
    paper_bgcolor='#0d1b2a', plot_bgcolor='#111f30',
    font=dict(color='#e2e8f0'),
)
fig.show()

---
## Going further — xsuite for complex lattices

For production-grade simulations with realistic lattice files (MAD-X, BMAD, SAD), **[xsuite](https://github.com/xsuite/xsuite)** is the standard at CERN. It handles:
- Full 6D symplectic tracking
- Space charge
- Synchrotron radiation
- Import from MAD-X SEQL/TWISS files

A minimal xsuite example for comparison:

```python
import xtrack as xt

# Load from MAD-X lattice file
line = xt.Line.from_madx_seqfile('lattice.seq', sequence='ring')
line.build_tracker()

# Create particles
particles = line.build_particles(x_norm=np.random.randn(1000), nemitt_x=1e-6)

# Track
line.track(particles, num_turns=1)

# Beam properties
sigma_x = particles.x.std()   # m
```

Use **Cheetah** when you need differentiability and ML integration.
Use **xsuite** when you need realistic lattice import and production-grade physics.

---
## Next steps
- Extend the lattice (add more quads, correctors) and see how BO scales
- Add noise to the objective (simulate measurement noise) — BO handles this naturally
- Add constraints (e.g. max beam size at intermediate points)
- **Notebook 03** → Train a neural network surrogate to replace Cheetah in the optimisation loop